# 17 — Final Model Training and Artifact Preparation

This notebook finalizes the selected fraud detection model workflow and prepares the project for inference and FastAPI integration. The focus is production readiness: clear paths, final artifact locations, and a clean setup for training and saving the validated model in later phases.


## Purpose

This notebook exists to prepare the final training and artifact-saving workflow for the fraud detection system.

It will eventually be responsible for:

- Training the final selected fraud detection model
- Reusing the final decision policy from previous notebooks
- Saving model artifacts needed for inference
- Saving feature columns, metrics, metadata, and decision policy
- Preparing the project for the next inference/API stage

This notebook should not repeat full EDA, full model comparison, or threshold tuning. Those tasks were completed earlier in the project. Notebook 17 is focused on turning the validated modeling decisions into reusable production-style artifacts.


## Previous Notebook Context

The project workflow leading into this notebook is:

- `13_model_training.ipynb` trained candidate models.
- `14_model_evaluation.ipynb` evaluated model performance and confirmed the strongest model choice.
- `15_threshold_tuning.ipynb` selected suitable fraud probability thresholds using validation/OOF predictions and holdout confirmation.
- `16_decision_logic.ipynb` converted model fraud probabilities into `APPROVE`, `REVIEW`, and `BLOCK` decisions.
- `17_final_model_training.ipynb` now prepares the final validated model training and artifact-saving workflow.

The difference between notebook 13 and notebook 17 is important. Notebook 13 is for experimentation and candidate model training. Notebook 17 is for final validated model training and artifact preparation for inference/API use. In other words, notebook 13 helps decide what works; notebook 17 prepares the final version for reuse by downstream code.


## Imports

This section imports the libraries needed for the notebook setup and the later final-training workflow. The imports are intentionally placed up front so the notebook can train the final model, compute metrics, and save artifacts without changing the environment configuration.


In [39]:
import json
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


## Path Configuration

This section defines all important input and output paths with `pathlib.Path`. The project root logic works whether the notebook is run from the repository root or from inside the `notebooks/` folder.

This setup step only configures paths. It does not load the dataset, train a model, evaluate a model, or save final model artifacts.


In [40]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
REPORTS_DIR = PROJECT_ROOT / "reports"
TABLES_DIR = REPORTS_DIR / "tables" / "17_final_model_training"
FIGURES_DIR = REPORTS_DIR / "figures" / "17_final_model_training"

# Existing processed dataset selected for final model training setup.
# The repository currently contains final_features.csv rather than creditcard_model_ready.csv.
FINAL_DATASET_PATH = DATA_PROCESSED_DIR / "final_features.csv"
DECISION_POLICY_PATH = ARTIFACTS_DIR / "decision_policy.json"

FINAL_MODEL_PATH = ARTIFACTS_DIR / "final_validated_fraud_model.joblib"
FINAL_FEATURE_COLUMNS_PATH = ARTIFACTS_DIR / "final_feature_columns.json"
FINAL_MODEL_METADATA_PATH = ARTIFACTS_DIR / "final_model_metadata.json"
FINAL_MODEL_METRICS_PATH = ARTIFACTS_DIR / "final_model_metrics.json"
FINAL_DECISION_POLICY_PATH = ARTIFACTS_DIR / "final_decision_policy.json"


## Output Folder Creation

This section creates the folders that later phases will use for artifacts, tables, and figures. Creating them now makes the notebook environment easy to verify before any final model training begins.


In [41]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

configured_paths = {
    "Project root": PROJECT_ROOT,
    "Processed data directory": DATA_PROCESSED_DIR,
    "Artifacts directory": ARTIFACTS_DIR,
    "Tables output directory": TABLES_DIR,
    "Figures output directory": FIGURES_DIR,
    "Final dataset path": FINAL_DATASET_PATH,
    "Decision policy path": DECISION_POLICY_PATH,
    "Final model artifact path": FINAL_MODEL_PATH,
}

print("Configured notebook paths:")
for label, path in configured_paths.items():
    print(f"- {label}: {path}")


Configured notebook paths:
- Project root: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection
- Processed data directory: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/data/processed
- Artifacts directory: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts
- Tables output directory: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/reports/tables/17_final_model_training
- Figures output directory: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/reports/figures/17_final_model_training
- Final dataset path: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/data/processed/final_features.csv
- Decision policy path: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/decision_policy.json
- Final model artifact path: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_validated_fraud_model.joblib


## Environment Check

Before moving into dataset loading and final model training, confirm the setup questions below:

- Do I know the input dataset path?
- Do I know where final model artifacts will be saved?
- Do I know where reports and tables will be saved?
- Do I understand the difference between notebook 13 and notebook 17?
- Did the required output folders get created successfully?

At this point the notebook environment is configured, but no data has been loaded, no model has been trained, no evaluation has been run, and no final model artifacts have been saved yet.


## Final Model-Ready Dataset Intake

This section loads the final processed dataset that was prepared in earlier notebooks. The purpose is to confirm that the dataset is available and to separate input features from the target column before final model training.


### Load Dataset

The final model-ready dataset should come from the processed data directory configured earlier in this notebook. This section checks that the path exists before loading anything. If the file is missing, the notebook shows the available processed CSV files and stops with a clear error so the dataset path can be corrected safely.


In [42]:
if not FINAL_DATASET_PATH.exists():
    available_files = list(DATA_PROCESSED_DIR.glob("*.csv"))
    print("Available processed CSV files:")
    for file in available_files:
        print("-", file.name)
    raise FileNotFoundError(f"Final dataset not found: {FINAL_DATASET_PATH}")

df = pd.read_csv(FINAL_DATASET_PATH)


### Dataset Shape

This section confirms the size of the final training dataset and shows the first few rows. It helps verify that the notebook is pointing to the expected processed file before any final training logic is added.


In [43]:
print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")
display(df.head())


Dataset shape: 283726 rows x 14 columns


,V14_V12_interaction,V14,V17_V16_interaction,V12,V17,V10,V4,V16,V3,V11,V7,V18,log_amount,Class
0,0.192241,-0.311169,-0.097830,-0.617801,0.207971,0.090794,1.378155,-0.470401,2.536347,-0.551600,0.239599,0.025791,5.014760,0
1,-0.153151,-0.143772,-0.053260,1.065235,-0.114805,-0.166974,0.448154,0.463917,0.166480,1.612727,-0.078803,-0.183361,1.305626,0
2,-0.010966,-0.165946,-3.207904,0.066084,1.109969,0.207643,0.379780,-2.890083,1.773209,0.624501,0.791461,-0.121359,5.939276,0
3,-0.051316,-0.287924,0.724897,0.178228,-0.684093,-0.054952,-0.863291,-1.059647,1.792993,-0.226487,0.237609,1.965775,4.824306,0
4,-0.602601,-1.119670,0.107008,0.538196,-0.237033,0.753074,0.403034,-0.451449,1.548718,-0.822843,0.592941,-0.038195,4.262539,0


### Identify Target Column

The fraud target is expected to be stored in the `Class` column.

- `Class = 1` means a fraud transaction.
- `Class = 0` means a normal or non-fraud transaction.

This check keeps the notebook explicit about the prediction target and avoids silent mistakes if the processed dataset schema changes.


In [44]:
TARGET_COL = "Class"

if TARGET_COL not in df.columns:
    print("Available columns:")
    print(df.columns.tolist())
    raise ValueError(f"Target column '{TARGET_COL}' not found in dataset.")


### Separate Features and Target

The final model will eventually learn from the feature matrix `X` to predict the target vector `y`. This phase only performs the separation and prints a compact summary. No train/test split, training, or evaluation happens yet.


In [45]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Number of feature columns: {X.shape[1]}")
print("First 10 feature columns:")
print(X.columns[:10].tolist())
print("Target distribution:")
print(y.value_counts())


Feature matrix shape: (283726, 13)
Target shape: (283726,)
Number of feature columns: 13
First 10 feature columns:
['V14_V12_interaction', 'V14', 'V17_V16_interaction', 'V12', 'V17', 'V10', 'V4', 'V16', 'V3', 'V11']
Target distribution:
Class
0    283253
1       473
Name: count, dtype: int64


## Dataset Understanding Summary

- `X` contains the input features used by the model.
- `y` contains the correct fraud and non-fraud labels.
- The model will eventually learn patterns from `X` to predict `y`.
- `Class = 1` means fraud.
- `Class = 0` means a normal transaction.
- This step does not validate the full dataset yet; detailed validation can be handled in the next notebook section you choose to add.


## Final Training Dataset Validation

Before final model training, the dataset must be validated to confirm that it is complete, correctly labeled, and structurally consistent. This is important because the final saved model will later be used by the inference pipeline and FastAPI service.


### Missing Value Check

Missing values can break model training or cause the model to learn from incomplete records. For final model training, the dataset should ideally have no missing values.


In [46]:
total_missing_values = int(df.isna().sum().sum())
missing_values_by_column = df.isna().sum()
columns_with_missing_values = missing_values_by_column[missing_values_by_column > 0].sort_values(ascending=False)

print(f"Total missing values: {total_missing_values}")
if columns_with_missing_values.empty:
    print("No missing values found in the final training dataset.")
else:
    display(columns_with_missing_values.rename("missing_value_count").to_frame())


Total missing values: 0
No missing values found in the final training dataset.


### Duplicate Row Check

Duplicate rows can make the model over-learn repeated examples. In a final training dataset, duplicates should be understood before training.

Exact duplicate rows were found in the final training dataset. Since this notebook prepares the final validated model for inference and API usage, exact duplicates are removed before train/test splitting. This reduces the risk that identical rows appear in both training and test data, which could make evaluation results look better than they really are.

The original dataframe `df` is kept unchanged for reporting. A new dataframe `df_model` is created and used for final model training.


In [47]:
duplicate_rows = int(df.duplicated().sum())
duplicate_percentage = (duplicate_rows / len(df) * 100) if len(df) else 0.0
duplicate_rows_full = df[df.duplicated(keep=False)]
duplicate_rows_full_count = int(len(duplicate_rows_full))
duplicate_rows_by_target = duplicate_rows_full[TARGET_COL].value_counts().sort_index()
target_counts_original = df[TARGET_COL].value_counts().sort_index()
duplicate_rows_percentage_by_target = (
    duplicate_rows_by_target / target_counts_original.reindex(duplicate_rows_by_target.index, fill_value=0) * 100
)

df_model = df.drop_duplicates().copy()
rows_removed = int(df.shape[0] - df_model.shape[0])
percentage_removed = ((rows_removed / df.shape[0]) * 100) if len(df) else 0.0

X = df_model.drop(columns=[TARGET_COL])
y = df_model[TARGET_COL]
fraud_count = int(y.sum())
normal_count = int(len(y) - fraud_count)
fraud_percentage = (fraud_count / len(y) * 100) if len(y) else 0.0
target_counts = y.value_counts().sort_index()
normal_percentage = (normal_count / len(y) * 100) if len(y) else 0.0
imbalance_ratio = (normal_count / fraud_count) if fraud_count else float("inf")

print(f"Duplicate rows: {duplicate_rows}")
print(f"Duplicate percentage: {duplicate_percentage:.2f}%")
print(f"Total duplicate rows including all repeated copies: {duplicate_rows_full_count}")
print("Duplicate rows by target class:")
print(duplicate_rows_by_target)
print("Duplicate rows percentage by target class:")
print(duplicate_rows_percentage_by_target)
print(f"Original dataset shape: {df.shape}")
print(f"Dataset shape after duplicate removal: {df_model.shape}")
print(f"Rows removed: {rows_removed}")
print(f"Percentage removed: {percentage_removed:.2f}%")
print(f"Feature matrix shape after duplicate removal: {X.shape}")
print(f"Target shape after duplicate removal: {y.shape}")
print("Target distribution after duplicate removal:")
print(y.value_counts())
print(f"Fraud cases after duplicate removal: {fraud_count}")
print(f"Normal cases after duplicate removal: {normal_count}")
print(f"Fraud percentage after duplicate removal: {fraud_percentage:.4f}%")


Duplicate rows: 8063
Duplicate percentage: 2.84%
Total duplicate rows including all repeated copies: 12446
Duplicate rows by target class:
Class
0    12446
Name: count, dtype: int64
Duplicate rows percentage by target class:
Class
0    4.393952
Name: count, dtype: float64
Original dataset shape: (283726, 14)
Dataset shape after duplicate removal: (275663, 14)
Rows removed: 8063
Percentage removed: 2.84%
Feature matrix shape after duplicate removal: (275663, 13)
Target shape after duplicate removal: (275663,)
Target distribution after duplicate removal:
Class
0    275190
1       473
Name: count, dtype: int64
Fraud cases after duplicate removal: 473
Normal cases after duplicate removal: 275190
Fraud percentage after duplicate removal: 0.1716%


### Target Label Validation

The target column must contain only valid binary fraud labels. The expected values are `0` for a normal or non-fraud transaction and `1` for a fraud transaction.


In [48]:
valid_target_values = {0, 1}
actual_target_values = {int(value) for value in df[TARGET_COL].dropna().unique()}
unexpected_values = actual_target_values - valid_target_values

print(f"Unique target values: {sorted(actual_target_values)}")
if unexpected_values:
    raise ValueError(f"Unexpected target values found: {unexpected_values}")

print("Target labels are valid. Only 0 and 1 are present.")


Unique target values: [0, 1]
Target labels are valid. Only 0 and 1 are present.


### Target Distribution and Class Imbalance Summary

Credit card fraud detection is highly imbalanced because fraud transactions are rare compared to normal transactions. This is why accuracy alone is not enough for evaluating the model.


In [49]:
total_records = len(y)
normal_percentage = (normal_count / total_records * 100) if total_records else 0.0

target_distribution_summary = pd.DataFrame(
    [
        {
            "class_label": 0,
            "class_meaning": "Normal Transaction",
            "count": normal_count,
            "percentage": normal_percentage,
        },
        {
            "class_label": 1,
            "class_meaning": "Fraud Transaction",
            "count": fraud_count,
            "percentage": fraud_percentage,
        },
    ]
)

display(target_distribution_summary)
print(f"Fraud cases: {fraud_count}")
print(f"Normal cases: {normal_count}")
print(f"Fraud percentage: {fraud_percentage:.4f}%")
print(f"Class imbalance ratio: {imbalance_ratio:.2f} : 1")


,class_label,class_meaning,count,percentage
0,0,Normal Transaction,275190,99.828414
1,1,Fraud Transaction,473,0.171586


Fraud cases: 473
Normal cases: 275190
Fraud percentage: 0.1716%
Class imbalance ratio: 581.80 : 1


### Feature Column Consistency Check

The final training feature matrix should contain only model input columns. The target column must already be removed before any model training starts.


In [50]:
if TARGET_COL in X.columns:
    raise ValueError(f"Target column '{TARGET_COL}' is still present in X.")

feature_column_table = pd.DataFrame(
    {
        "feature_index": range(len(X.columns)),
        "feature_name": X.columns,
    }
)

print(f"Number of feature columns: {X.shape[1]}")
print("Target column is not present in X. Feature matrix is correctly separated.")
display(feature_column_table)


Number of feature columns: 13
Target column is not present in X. Feature matrix is correctly separated.


,feature_index,feature_name
0,0,V14_V12_interaction
1,1,V14
2,2,V17_V16_interaction
3,3,V12
4,4,V17
5,5,V10
6,6,V4
7,7,V16
8,8,V3
9,9,V11


### Data Type Check

For this fraud dataset, model features should be numeric. This check confirms that every feature column inside `X` is suitable for model training without additional type conversion.


In [51]:
feature_dtype_table = X.dtypes.rename("dtype").reset_index().rename(columns={"index": "feature_name"})
numeric_feature_columns = X.select_dtypes(include=[np.number]).columns.tolist()
non_numeric_feature_columns = X.select_dtypes(exclude=[np.number]).columns.tolist()

display(feature_dtype_table)
print(f"Numeric feature columns: {len(numeric_feature_columns)}")
print(f"Non-numeric feature columns: {len(non_numeric_feature_columns)}")

if non_numeric_feature_columns:
    print("Non-numeric feature columns detected:")
    print(non_numeric_feature_columns)
    raise TypeError(
        "Non-numeric feature columns found in X. Final model training requires numeric features only."
    )

print("All feature columns are numeric and ready for model training.")


,feature_name,dtype
0,V14_V12_interaction,float64
1,V14,float64
2,V17_V16_interaction,float64
3,V12,float64
4,V17,float64
5,V10,float64
6,V4,float64
7,V16,float64
8,V3,float64
9,V11,float64


Numeric feature columns: 13
Non-numeric feature columns: 0
All feature columns are numeric and ready for model training.


### Dataset Readiness Summary

This summary collects the most important validation checks into one compact table. It is also saved to the reporting folder so the final training workflow has a documented validation checkpoint. The summary distinguishes between the original reporting dataset and the deduplicated dataset used for final model training.


In [52]:
all_features_numeric = len(non_numeric_feature_columns) == 0
target_column_removed_from_X = TARGET_COL not in X.columns
validation_summary_path = TABLES_DIR / "final_training_dataset_validation_summary.csv"

dataset_validation_summary = pd.DataFrame(
    [
        {"check": "original_dataset_rows", "value": df.shape[0]},
        {"check": "original_dataset_columns", "value": df.shape[1]},
        {"check": "duplicate_rows_removed", "value": rows_removed},
        {"check": "duplicate_percentage_removed", "value": round(percentage_removed, 6)},
        {"check": "final_training_rows", "value": df_model.shape[0]},
        {"check": "final_training_columns", "value": df_model.shape[1]},
        {"check": "feature_columns_after_duplicate_removal", "value": X.shape[1]},
        {"check": "target_column", "value": TARGET_COL},
        {"check": "total_missing_values", "value": total_missing_values},
        {"check": "target_values", "value": str(sorted(actual_target_values))},
        {"check": "normal_transactions_after_duplicate_removal", "value": normal_count},
        {"check": "fraud_transactions_after_duplicate_removal", "value": fraud_count},
        {"check": "fraud_percentage_after_duplicate_removal", "value": round(fraud_percentage, 6)},
        {"check": "imbalance_ratio_after_duplicate_removal", "value": round(imbalance_ratio, 6) if np.isfinite(imbalance_ratio) else "inf"},
        {"check": "all_features_numeric", "value": all_features_numeric},
        {"check": "target_column_removed_from_X", "value": target_column_removed_from_X},
    ]
)

dataset_validation_summary.to_csv(validation_summary_path, index=False)
display(dataset_validation_summary)
print(f"Saved dataset validation summary to: {validation_summary_path}")


,check,value
0,original_dataset_rows,283726
1,original_dataset_columns,14
2,duplicate_rows_removed,8063
3,duplicate_percentage_removed,2.841826
4,final_training_rows,275663
5,final_training_columns,14
6,feature_columns_after_duplicate_removal,13
7,target_column,Class
8,total_missing_values,0
9,target_values,"[0, 1]"


Saved dataset validation summary to: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/reports/tables/17_final_model_training/final_training_dataset_validation_summary.csv


## Final Training Dataset Validation Notes

The dataset is considered ready for final model training if:

- Missing values are not present
- Target contains only valid labels 0 and 1
- Exact duplicate rows are removed before train/test splitting by creating `df_model`
- Feature matrix does not contain the target column
- Feature columns are numeric
- Class imbalance is clearly understood and will be considered during model training and evaluation


## Stop and Verify Before Model Training

- Were exact duplicate rows found?
- How many duplicate rows were removed?
- Was the original `df` kept unchanged for reporting?
- Is `df_model` now the dataframe used for final model training?
- Were `X` and `y` recreated from `df_model`?
- Did fraud percentage change after duplicate removal?
- Is the dataset still highly imbalanced after duplicate removal?


## Final Decision Policy Loading

The model will output a fraud probability, but the business system needs a final decision such as `APPROVE`, `REVIEW`, or `BLOCK`. The thresholds for these decisions were already selected in the threshold tuning and decision logic notebooks. This notebook only loads those saved rules so the final trained model can use the same decision policy during validation and inference.

This notebook does not create new thresholds and does not tune thresholds again. It only reuses the approved policy from the earlier workflow.


### Load Decision Policy File

The saved decision policy should already exist in the artifacts directory. This section checks that the file is present before loading it. If the file is missing, the notebook prints the available JSON files and stops with a clear error so the artifact path can be corrected safely.


In [53]:
if not DECISION_POLICY_PATH.exists():
    available_policy_files = list(ARTIFACTS_DIR.glob("*.json"))
    print("Available JSON files in artifacts directory:")
    for file in available_policy_files:
        print(f"- {file.name}")
    raise FileNotFoundError(
        f"Decision policy file not found: {DECISION_POLICY_PATH}. "
        "Please make sure 16_decision_logic.ipynb has saved decision_policy.json."
    )

with open(DECISION_POLICY_PATH, "r", encoding="utf-8") as f:
    decision_policy = json.load(f)

print("Decision policy loaded successfully.")
decision_policy


Decision policy loaded successfully.


{'model_name': 'Random Forest',
 'review_threshold': 0.35,
 'block_threshold': 0.5,
 'block_threshold_candidate': 0.821,
 'decision_policy': {'APPROVE': 'fraud_probability < review_threshold',
  'REVIEW': 'review_threshold <= fraud_probability < block_threshold',
  'BLOCK': 'fraud_probability >= block_threshold'},
 'risk_levels': {'Very High Risk': 'fraud_probability >= block_threshold',
  'High Risk': 'max(block_threshold - 0.10, review_threshold) <= fraud_probability < block_threshold',
  'Medium Risk': 'review_threshold <= fraud_probability < max(block_threshold - 0.10, review_threshold)',
  'Low-Medium Risk': '0.10 <= fraud_probability < review_threshold',
  'Low Risk': 'fraud_probability < 0.10'},
 'source_threshold_artifact': 'artifacts/selected_threshold.json',
 'source_prediction_file': 'reports/tables/13_model_training/baseline_test_predictions.csv',
 'intended_use': 'FastAPI fraud decision response'}

### Extract and Validate Decision Thresholds

The notebook expects a review threshold and a block threshold from the saved decision policy. The extraction logic below supports both flat and nested JSON layouts so the notebook can read the artifact safely without rewriting earlier project decisions.


In [54]:
if "review_threshold" in decision_policy and "block_threshold" in decision_policy:
    review_threshold = decision_policy["review_threshold"]
    block_threshold = decision_policy["block_threshold"]
elif "thresholds" in decision_policy:
    review_threshold = decision_policy["thresholds"].get("review_threshold")
    block_threshold = decision_policy["thresholds"].get("block_threshold")
else:
    raise KeyError(
        "Decision policy must contain review_threshold and block_threshold."
    )

if review_threshold is None or block_threshold is None:
    raise ValueError("Both review_threshold and block_threshold are required.")

if not isinstance(review_threshold, (int, float)) or not isinstance(block_threshold, (int, float)):
    raise TypeError("Decision thresholds must be numeric values.")

review_threshold = float(review_threshold)
block_threshold = float(block_threshold)

if not 0 <= review_threshold <= 1:
    raise ValueError("review_threshold must be between 0 and 1.")

if not 0 <= block_threshold <= 1:
    raise ValueError("block_threshold must be between 0 and 1.")

if review_threshold >= block_threshold:
    raise ValueError("review_threshold must be less than block_threshold.")

thresholds_valid = True
print(f"Review threshold: {review_threshold}")
print(f"Block threshold: {block_threshold}")
print("Decision thresholds are valid.")


Review threshold: 0.35
Block threshold: 0.5
Decision thresholds are valid.


### Validate Decision Labels

The final business workflow expects three labels: `APPROVE`, `REVIEW`, and `BLOCK`. If the artifact already defines them, the notebook validates those labels. If not, it creates the standard local mapping used across the project.


In [55]:
required_decision_labels = {"APPROVE", "REVIEW", "BLOCK"}
policy_decision_rules = decision_policy.get("decision_policy")

if isinstance(policy_decision_rules, dict):
    actual_labels = set(policy_decision_rules.keys())
    if actual_labels != required_decision_labels:
        raise ValueError(
            f"Decision policy labels must be exactly {sorted(required_decision_labels)}, "
            f"but found {sorted(actual_labels)}."
        )
    decision_labels = {label.lower(): label for label in sorted(actual_labels)}
else:
    decision_labels = {
        "approve": "APPROVE",
        "review": "REVIEW",
        "block": "BLOCK",
    }

final_labels = {decision_labels["approve"], decision_labels["review"], decision_labels["block"]}
if final_labels != required_decision_labels:
    raise ValueError(
        f"Final decision labels must be exactly {sorted(required_decision_labels)}."
    )

print("Decision labels:")
for key in ["approve", "review", "block"]:
    print(f"- {key}: {decision_labels[key]}")


Decision labels:
- approve: APPROVE
- review: REVIEW
- block: BLOCK


### Decision Rule Summary

This table translates the fraud probability thresholds into business actions. It documents exactly how the final system will move from a numeric model score to an operational decision.


In [56]:
decision_rule_summary = pd.DataFrame(
    [
        {
            "decision": "APPROVE",
            "condition": f"probability < {review_threshold:.3f}",
            "meaning": "Transaction is considered low risk and can be approved.",
        },
        {
            "decision": "REVIEW",
            "condition": (
                f"{review_threshold:.3f} <= probability < {block_threshold:.3f}"
            ),
            "meaning": "Transaction requires manual or secondary review.",
        },
        {
            "decision": "BLOCK",
            "condition": f"probability >= {block_threshold:.3f}",
            "meaning": "Transaction is considered high risk and should be blocked.",
        },
    ]
)

decision_policy_summary_path = TABLES_DIR / "final_decision_policy_summary.csv"
decision_rule_summary.to_csv(decision_policy_summary_path, index=False)
display(decision_rule_summary)
print(f"Saved decision policy summary to: {decision_policy_summary_path}")


,decision,condition,meaning
0,APPROVE,probability < 0.350,Transaction is considered low risk and can be ...
1,REVIEW,0.350 <= probability < 0.500,Transaction requires manual or secondary review.
2,BLOCK,probability >= 0.500,Transaction is considered high risk and should...


Saved decision policy summary to: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/reports/tables/17_final_model_training/final_decision_policy_summary.csv


### Decision Logic Explanation

The model produces a fraud probability between `0` and `1`. The decision policy converts this probability into a business action.

- If probability is below `review_threshold`, the decision is `APPROVE`.
- If probability is greater than or equal to `review_threshold` but below `block_threshold`, the decision is `REVIEW`.
- If probability is greater than or equal to `block_threshold`, the decision is `BLOCK`.

For example, if `block_threshold` is `0.50` and the model gives a fraud probability of `0.72`, the final decision is `BLOCK`.


### Reusable Decision Function

This small helper function will be reused later when the final model generates probabilities. For now, the notebook only tests the function with example values. It is not applied to real model predictions in this section because the final model has not been trained yet.


In [57]:
def assign_decision(probability, review_threshold, block_threshold):
    """Convert a fraud probability into an APPROVE, REVIEW, or BLOCK decision."""
    if not 0 <= probability <= 1:
        raise ValueError("probability must be between 0 and 1.")
    if probability >= block_threshold:
        return "BLOCK"
    if probability >= review_threshold:
        return "REVIEW"
    return "APPROVE"

example_probabilities = [
    0.05,
    review_threshold,
    (review_threshold + block_threshold) / 2,
    block_threshold,
    0.90,
]

decision_function_test_df = pd.DataFrame(
    {
        "fraud_probability": example_probabilities,
        "assigned_decision": [
            assign_decision(probability, review_threshold, block_threshold)
            for probability in example_probabilities
        ],
    }
)

display(decision_function_test_df)


,fraud_probability,assigned_decision
0,0.050,APPROVE
1,0.350,REVIEW
2,0.425,REVIEW
3,0.500,BLOCK
4,0.900,BLOCK


### Decision Policy Readiness Summary

This summary records that the notebook loaded the approved decision policy successfully and extracted the thresholds needed for later training validation and inference artifacts.


In [58]:
decision_policy_readiness_summary = pd.DataFrame(
    [
        {
            "check": "policy_file_path",
            "value": str(DECISION_POLICY_PATH),
        },
        {
            "check": "review_threshold",
            "value": review_threshold,
        },
        {
            "check": "block_threshold",
            "value": block_threshold,
        },
        {
            "check": "approve_label",
            "value": decision_labels["approve"],
        },
        {
            "check": "review_label",
            "value": decision_labels["review"],
        },
        {
            "check": "block_label",
            "value": decision_labels["block"],
        },
        {
            "check": "policy_loaded_successfully",
            "value": True,
        },
        {
            "check": "thresholds_valid",
            "value": thresholds_valid,
        },
        {
            "check": "decision_function_tested",
            "value": True,
        },
    ]
)

decision_policy_readiness_summary_path = (
    TABLES_DIR / "final_decision_policy_readiness_summary.csv"
)
decision_policy_readiness_summary.to_csv(
    decision_policy_readiness_summary_path, index=False
)
display(decision_policy_readiness_summary)
print(
    f"Saved decision policy readiness summary to: {decision_policy_readiness_summary_path}"
)


,check,value
0,policy_file_path,/Users/mohammadmubashir/VCode/Credit-Card-Frau...
1,review_threshold,0.35
2,block_threshold,0.5
3,approve_label,APPROVE
4,review_label,REVIEW
5,block_label,BLOCK
6,policy_loaded_successfully,True
7,thresholds_valid,True
8,decision_function_tested,True


Saved decision policy readiness summary to: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/reports/tables/17_final_model_training/final_decision_policy_readiness_summary.csv


## Stop and Verify Before Training

- Was `decision_policy.json` loaded successfully?
- Do I know where the policy file came from?
- Do I understand that this notebook is reusing thresholds, not creating new ones?
- What does `review_threshold` mean?
- What does `block_threshold` mean?
- Is `review_threshold` lower than `block_threshold`?
- Are the final decision labels `APPROVE`, `REVIEW`, and `BLOCK`?
- Did the sample decision function return expected results?
